In [ ]:
import os

# Force Python path
os.environ["PYSPARK_PYTHON"] = r"c:\Users\Administrator\Desktop\DE\venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"c:\Users\Administrator\Desktop\DE\venv\Scripts\python.exe"

# CRITICAL FIX (network binding)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.host=127.0.0.1 --conf spark.driver.bindAddress=127.0.0.1 pyspark-shell"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RDD Impl")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")
    .getOrCreate()
)

# Get the SparkContext
sc = spark.sparkContext

# Create RDD
rdd = sc.parallelize([1, 2, 3, 4, 5])

print(rdd.collect())


In [ ]:
spark.stop()

In [ ]:
import os

# Update to use the venv on this machine
os.environ["PYSPARK_PYTHON"] = r"c:\Users\Administrator\Desktop\DE\venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"c:\Users\Administrator\Desktop\DE\venv\Scripts\python.exe"

# Stronger network fix
os.environ["PYSPARK_SUBMIT_ARGS"] = """
--conf spark.driver.host=127.0.0.1 
--conf spark.driver.bindAddress=127.0.0.1 
--conf spark.python.worker.reuse=false 
--conf spark.network.timeout=600s 
--conf spark.executor.heartbeatInterval=60s 
pyspark-shell
"""


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StableRDD") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.network.timeout", "600s") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
#Basic RDD Transformations
rdd = sc.parallelize([1, 2, 3, 4])
squared = rdd.map(lambda x: x * x)
print(squared.collect())

In [ ]:
#filter
even = rdd.filter(lambda x: x % 2 == 0)

print(even.collect())

In [ ]:
#FlatMap
data = ["hello world", "spark is powerful"]

rdd = sc.parallelize(data)

words = rdd.flatMap(lambda x: x.split(" "))

print(words.collect())

In [ ]:
#Key-Value RDD Operations
data = [("a", 1), ("b", 2), ("a", 3), ("b", 4)]

rdd = sc.parallelize(data)

In [ ]:
#reduceByKey()
result = rdd.reduceByKey(lambda x, y: x + y)

print(result.collect())

In [ ]:
#groupByKey() (costly)
result = rdd.groupByKey().mapValues(list)

print(result.collect())

In [ ]:
rdd = sc.parallelize([10, 20, 30])

print(rdd.count())
print(rdd.first())
print(rdd.take(2))
print(rdd.reduce(lambda x, y: x + y))

In [ ]:
#counting the number of words and max words
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StableRDD") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.network.timeout", "600s") \
    .getOrCreate()

sc = spark.sparkContext

# Load text file
rdd = sc.textFile("word_count_text.txt")

# Split into words
words = rdd.flatMap(lambda line: line.lower().split())

# Remove punctuation (basic cleaning)
import re
words_clean = words.map(lambda word: re.sub(r'[^a-z]', '', word)) \
                   .filter(lambda word: word != "")

# Word Count
word_count = words_clean.map(lambda word: (word, 1)) \
                        .reduceByKey(lambda a, b: a + b)

print("Word Count")
word_count.collect()

# Max Repeated Word
max_word = word_count.reduce(lambda a, b: a if a[1] > b[1] else b)

print("=== Most Repeated Word ===")
print(max_word)

#TRY
#Find longest word
#Find average word length

In [ ]:
#Top 3 Products by Total Sales
rdd = sc.textFile("sales_data.csv")

# Remove header
header = rdd.first()
data = rdd.filter(lambda x: x != header)

# Parse → (product, amount)
product_sales = data.map(lambda x: x.split(",")) \
                    .map(lambda x: (x[1], int(x[2])))

# Aggregate
total_sales = product_sales.reduceByKey(lambda x, y: x + y)

# Sort descending and take top 3
top3 = total_sales.sortBy(lambda x: x[1], ascending=False).take(3)

print(top3)

In [ ]:
#Second Highest Sale Per Country
rdd = sc.textFile("sales_data.csv")

header = rdd.first()
data = rdd.filter(lambda x: x != header)

parsed = data.map(lambda x: x.split(",")) \
             .map(lambda x: (x[0], int(x[2])))

# Group values
grouped = parsed.groupByKey()

# Find second highest
second_highest = grouped.mapValues(lambda vals: sorted(vals, reverse=True)[1] if len(vals) > 1 else None)

print(second_highest.collect())

In [ ]:
#Remove Duplicates WITHOUT distinct()
rdd = sc.parallelize([1,2,2,3,4,4,5])

unique = rdd.map(lambda x: (x, 1)) \
            .reduceByKey(lambda x, y: x) \
            .map(lambda x: x[0])

print(unique.collect())

In [ ]:
#Total Spending Per User
import json

rdd = sc.textFile("nested_data.json")

data = rdd.map(lambda x: json.loads(x))

# Flatten
user_spend = data.flatMap(lambda user: [
    (user["user"], item["price"])
    for order in user["orders"]
    for item in order["items"]
])

# Aggregate
total = user_spend.reduceByKey(lambda x, y: x + y)

print(total.collect())

In [ ]:
##Total Spending Per User
import json

rdd = sc.wholeTextFiles("nested_data.json")

# Extract content
data = rdd.flatMap(lambda x: json.loads(x[1]))

# Now continue
user_spend = data.flatMap(lambda user: [
    (user["user"], item["price"])
    for order in user["orders"]
    for item in order["items"]
])

total = user_spend.reduceByKey(lambda x, y: x + y)

print(total.collect())

In [ ]:
Question 5: Most Sold Product (Global)

In [ ]:
rdd = sc.textFile("sales_data.csv")

header = rdd.first()
data = rdd.filter(lambda x: x != header)

product_count = data.map(lambda x: x.split(",")[1]) \
                    .map(lambda x: (x, 1)) \
                    .reduceByKey(lambda x, y: x + y)

top_product = product_count.sortBy(lambda x: x[1], ascending=False).first()

print(top_product)

In [ ]:
#Question : Join Two RDDs

In [ ]:
users = sc.parallelize([
    ("u1", "Nishanth"),
    ("u2", "John")
])

transactions = sc.parallelize([
    ("u1", 100),
    ("u2", 200),
    ("u1", 50)
])

joined = users.join(transactions)

print(joined.collect())

In [ ]:
#Question : Detect Data Skew

In [ ]:
#Detect Data Skew
rdd = sc.textFile("sales_data.csv")

header = rdd.first()
data = rdd.filter(lambda x: x != header)

country_count = data.map(lambda x: x.split(",")[0]) \
                    .map(lambda x: (x, 1)) \
                    .reduceByKey(lambda x, y: x + y)

# Detect skew (threshold example: > 100)
skewed = country_count.filter(lambda x: x[1] > 100)

print(skewed.collect())

In [ ]:
from pyspark.sql import SparkSession
import re

# Create Spark Session
spark = SparkSession.builder.appName("WordAnalysisRDD").getOrCreate()
sc = spark.sparkContext

# Load file
rdd = sc.textFile("word_count_text.txt")

# Split into words
words = rdd.flatMap(lambda line: line.lower().split())

# Clean words (remove punctuation)
words_clean = words.map(lambda w: re.sub(r'[^a-z]', '', w)) \
                   .filter(lambda w: w != "")


#  Longest Word

longest_word = words_clean.reduce(lambda a, b: a if len(a) > len(b) else b)

print("Longest Word:", longest_word)
print("Length:", len(longest_word))



# 2 Average Word Length 
# Convert each word to (length, 1)
length_rdd = words_clean.map(lambda w: (len(w), 1))

# Sum lengths and counts
total_length, total_words = length_rdd.reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]))

average_length = total_length / total_words

print("Average Word Length:", average_length)

In [ ]:
# Create Spark Session
import os
from pyspark.sql import SparkSession

# Configure local venv path
os.environ["PYSPARK_PYTHON"] = r"c:\Users\Administrator\Desktop\DE\venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"c:\Users\Administrator\Desktop\DE\venv\Scripts\python.exe"

spark = (
    SparkSession.builder
    .appName("Read and Filter Employees")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")
    .getOrCreate()
)

sc = spark.sparkContext

# --- DataFrame & Spark SQL implementation ---
# df = spark.read.csv("employees_records.csv", header=True, inferSchema=True)
# df.createOrReplaceTempView("Employee")

# Execute SQL query and display results
# spark.sql("select * from Employee where salary > 7000").show()

# spark.sql("""select CAST(AVG(salary) AS INT) AS avg_salary FROM Employee""").show()
# spark.sql("SELECT CAST(AVG(salary) AS INT) AS avg_salary FROM Employee").show()



# --- RDD implementation (commented out) ---
# rdd = sc.textFile("employees_records.csv")
# header = rdd.first()
# filtered_rdd = (
#     rdd.filter(lambda line: line != header)
#     .map(lambda line: line.split(","))
#     .filter(lambda cols: int(cols[5]) > 7000)
# )
# filtered_rdd.collect()


In [ ]:
spark.sql("SELECT department, COUNT(*) AS emp_count FROM Employee GROUP BY department ").show()


In [ ]:
df1 = spark.read.csv("customers_dataset.csv", header=True, inferSchema=True)
df2 = spark.read.csv("products_dataset.csv", header=True, inferSchema=True)
df3 = spark.read.csv("orders_dataset.csv", header=True, inferSchema=True)
df1.createOrReplaceTempView("Customers")
df2.createOrReplaceTempView("Products")
df3.createOrReplaceTempView("Orders")




In [ ]:
spark.sql("select * from Customers").show()
# spark.sql("select * from Products").show()
# spark.sql("select * from Orders").show()

In [ ]:
spark.sql("SELECT product_id,product_name,category,price,stock_quantity FROM Products WHERE price > 40000 AND stock_quantity > 20").show()

In [ ]:
spark.sql("SELECT customer_id, SUM(sales_amount) AS total_sales FROM orders GROUP BY customer_id").show()

In [ ]:
spark.sql("""
    SELECT customer_id, total_sales
    FROM (
        SELECT customer_id, SUM(sales_amount) AS total_sales
        FROM Orders
        GROUP BY customer_id
    ) AS sub
    ORDER BY total_sales DESC
    LIMIT 1
""").show()


In [ ]:
spark.sql("SELECT product_id, SUM(quantity) AS total_quantity_sold FROM orders GROUP BY product_id ORDER BY total_quantity_sold DESC LIMIT 1").show()

In [ ]:
spark.sql("""SELECT category, ROUND(AVG(price), 2) AS avg_price
    FROM Products
    GROUP BY category
    ORDER BY avg_price DESC""").show()

In [ ]:
spark.sql("""SELECT customer_id,
SUM(sales_amount) AS total_purchase
FROM orders
GROUP BY customer_id
HAVING SUM(sales_amount) > 100000
""").show()

In [ ]:
spark.sql("""
    SELECT 
        o.order_id,
        c.customer_name,
        p.product_name,
        p.category,
        p.brand,
        o.quantity,
        o.sales_amount,
        o.order_date,
        o.order_status
    FROM Orders o
    JOIN Customers c ON o.customer_id = c.customer_id
    JOIN Products p ON o.product_id = p.product_id
""").show()


In [ ]:
spark.sql("""SELECT product_id, 
price FROM
Products ORDER
BY price 
DESC LIMIT 5""").show()

In [ ]:
spark.sql("""
    SELECT 
        c.customer_id, 
        c.customer_name, 
        ROUND(SUM(o.sales_amount), 2) AS total_spending
    FROM Orders o
    JOIN Customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name
    ORDER BY total_spending DESC
    LIMIT 3
""").show()


In [ ]:
spark.sql("""
    SELECT 
    c.membership_type, 
    COUNT(o.order_id) AS total_orders
    FROM Orders o
    JOIN Customers c ON o.customer_id = c.customer_id
    GROUP BY c.membership_type
    ORDER BY total_orders DESC
""").show()


In [ ]:
spark.sql("""
    SELECT c.customer_id, c.customer_name
    FROM Customers AS c
    LEFT JOIN Orders o ON c.customer_id = o.customer_id
    WHERE o.customer_id IS NULL
""").show()


In [ ]:
spark.sql("""
    SELECT 
        c.city,
        COUNT(o.order_id) AS total_orders, 
        ROUND(SUM(o.sales_amount), 2) AS total_spending 
    FROM Orders o 
    JOIN Customers c ON o.customer_id = c.customer_id 
    GROUP BY c.city
    ORDER BY total_spending DESC
""").show()


In [ ]:
spark.sql("""
    SELECT p.product_id, p.product_name
    FROM Products p
    LEFT JOIN Orders o ON p.product_id = o.product_id
    WHERE o.order_id IS NULL
""").show()


In [ ]:
spark.sql("""SELECT customer_id, UPPER(customer_name) AS customer_name FROM customers""").show()

In [ ]:
spark.sql("""SELECT CONCAT(customer_name, '-', city) AS customer_details FROM customers""").show(20, truncate=False)

In [ ]:
spark.sql("""SELECT
customer_id, customer_name,membership_type,credit_score,RANK() OVER(PARTITION BY membership_type 
ORDER BY credit_score DESC) AS credit_rank FROM customers""").show()

In [ ]:
spark.sql("""
    WITH customer_sales AS (
        SELECT 
            c.city,
            c.customer_id,
            c.customer_name,
            SUM(o.sales_amount) AS total_sales
        FROM customers c 
        JOIN orders o ON c.customer_id = o.customer_id 
        GROUP BY c.city, c.customer_id, c.customer_name
    ),

    ranked_customers AS (
        SELECT 
            *, 
            ROW_NUMBER() OVER(PARTITION BY city ORDER BY total_sales DESC) AS rn 
        FROM customer_sales
    )

    SELECT city, customer_id, customer_name, total_sales 
    FROM ranked_customers 
    WHERE rn = 1
""").show()


In [ ]:
spark.sql("""SELECT order_id, order_date,YEAR(order_date) AS year,
MONTH(order_date) AS month, DAY(order_date) AS day FROM orders""").show()

In [ ]:
spark.sql("""
    SELECT 
        order_id,
        order_date, 
        ship_date, 
        DATEDIFF(ship_date, order_date) AS shipping_days
    FROM Orders
""").show()


In [ ]:
spark.sql("""
    WITH product_sales AS (
        SELECT 
            p.category,
            p.product_id,
            p.product_name,
            ROUND(SUM(o.sales_amount), 2) AS total_sales
        FROM Orders o
        JOIN Products p ON o.product_id = p.product_id
        GROUP BY p.category, p.product_id, p.product_name
    ),

    ranked_products AS (
        SELECT 
            *,
            RANK() OVER (PARTITION BY category ORDER BY total_sales DESC) AS rnk
        FROM product_sales
    )

    SELECT 
        category,
        product_id,
        product_name,
        total_sales,
        rnk
    FROM ranked_products
    WHERE rnk <= 3
    ORDER BY category, rnk
""").show()


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("LogAnalysisDF").getOrCreate()

# Load data
df = spark.read.text("app_logs.txt")

# Split into columns
split_col = split(df["value"], " ")

logs = df.select(
    split_col.getItem(0).alias("date"),
    split_col.getItem(1).alias("time"),
    split_col.getItem(2).alias("level"),
    split_col.getItem(3).alias("user"),
    split_col.getItem(4).alias("action"),
    split_col.getItem(5).alias("status_or_item")
)

logs.show(truncate=False)
logs.createOrReplaceTempView("Logs")


#Most Active User
#Failed Login Attempts
#Users with Both Success & Failure
#Consecutive Failures
#Users Who Never Made a Purchase

In [ ]:
spark.sql("""
    SELECT user, COUNT(*) AS total_actions
    FROM Logs
    GROUP BY user
    ORDER BY total_actions DESC
    LIMIT 1
""").show()


In [ ]:
spark.sql("""
SELECT *
FROM LOGS
WHERE action = 'login' and status_or_item = 'failed'
""").show()

In [ ]:
spark.sql("""
    SELECT user
    FROM Logs
    GROUP BY user
    HAVING COUNT(CASE WHEN status_or_item = 'success' THEN 1 END) > 0
       AND COUNT(CASE WHEN status_or_item = 'failed' THEN 1 END) > 0
""").show()


In [ ]:
spark.sql("""
    WITH ranked_logs AS (
        SELECT 
            *,
            LAG(status_or_item, 1) OVER (PARTITION BY user ORDER BY date, time) AS prev_status
        FROM Logs
    )
    SELECT *
    FROM ranked_logs
    WHERE status_or_item = 'failed' AND prev_status = 'failed'
""").show()


In [ ]:
spark.sql("""
    SELECT DISTINCT user
    FROM Logs
    WHERE user NOT IN (
        SELECT DISTINCT user
        FROM Logs
        WHERE action = 'purchase'
    )
""").show()


In [17]:
import os
from pyspark.sql import SparkSession

# Configure local venv path
os.environ["PYSPARK_PYTHON"] = r"c:\Users\Administrator\Desktop\DE\venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"c:\Users\Administrator\Desktop\DE\venv\Scripts\python.exe"

spark = (
    SparkSession.builder
    .appName("Read and Filter Employees")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")
    .getOrCreate()
)

sc = spark.sparkContext

df1 = spark.read.csv("customers_dataset.csv", header=True, inferSchema=True)
df2 = spark.read.csv("products_dataset.csv", header=True, inferSchema=True)
df3 = spark.read.csv("orders_dataset.csv", header=True, inferSchema=True)
df1.createOrReplaceTempView("Customers")
df2.createOrReplaceTempView("Products")
df3.createOrReplaceTempView("Orders")




In [ ]:
spark.sql("""
    SELECT c.customer_id, c.customer_name, ROUND(SUM(o.sales_amount), 2) AS total_revenue
    FROM Orders o
    JOIN Customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name
    ORDER BY total_revenue DESC
    LIMIT 10
""").show()


In [ ]:
spark.sql("""
    SELECT 
        DATE_FORMAT(order_date, 'yyyy-MM') AS month,
        SUM(sales_amount) AS monthly_revenue
    FROM Orders
    GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
    ORDER BY month ASC
""").show()


In [ ]:
spark.sql("""
SELECT
    order_id,
    order_date,
    sales_amount,
    ROUND(SUM(sales_amount)
OVER (ORDER BY order_date,
order_id), 2) AS
running_total_revenue
    FROM orders
    ORDER BY order_date, order_id
""").show()

In [ ]:
spark.sql("""
    SELECT 
        customer_id,
        order_id,
        order_date,
        sales_amount AS current_purchase,
        LAG(sales_amount, 1) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS previous_purchase
    FROM Orders
""").show()


In [ ]:
spark.sql("""
    SELECT 
        customer_id,
        order_id,
        order_date,
        sales_amount AS current_purchase,
        LEAD(sales_amount, 1) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS next_purchase
    FROM Orders
""").show()


In [ ]:
spark.sql("""
    WITH customer_months AS (
        SELECT DISTINCT 
            customer_id,
            MONTH(order_date) AS order_month,
            YEAR(order_date) AS order_year
        FROM Orders
    ),
    retention AS (
        SELECT 
            customer_id,
            order_year,
            order_month,
            LAG(order_month, 1) OVER (PARTITION BY customer_id, order_year ORDER BY order_month) AS prev_month
        FROM customer_months
    )
    SELECT DISTINCT customer_id
    FROM retention
    WHERE order_month = prev_month + 1
""").show()


In [ ]:
spark.sql("""
    WITH product_sales AS (
        SELECT 
            p.category,
            p.product_name,
            ROUND(SUM(o.sales_amount), 2) AS total_sales
        FROM Orders o
        JOIN Products p ON o.product_id = p.product_id
        GROUP BY p.category, p.product_name
    ),
    ranked AS (
        SELECT 
            *,
            DENSE_RANK() OVER (PARTITION BY category ORDER BY total_sales DESC) AS rnk
        FROM product_sales
    )
    SELECT category, product_name, total_sales
    FROM ranked
    WHERE rnk = 1
""").show()


In [ ]:
spark.sql("""
    SELECT 
        ROUND(AVG(DATEDIFF(ship_date, order_date)), 2) AS avg_shipping_delay_days
    FROM Orders
""").show()


In [ ]:
spark.sql("""
    SELECT c.customer_id, c.customer_name, c.city
    FROM Customers c
    LEFT JOIN Orders o ON c.customer_id = o.customer_id
    WHERE o.customer_id IS NULL
""").show()


In [ ]:
spark.sql("""
    WITH customer_revenue AS (
        SELECT 
            c.customer_id,
            c.customer_name,
            ROUND(SUM(o.sales_amount), 2) AS total_revenue
        FROM Orders o
        JOIN Customers c ON o.customer_id = c.customer_id
        GROUP BY c.customer_id, c.customer_name
    )
    SELECT 
        customer_id,
        customer_name,
        total_revenue,
        DENSE_RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
    FROM customer_revenue
""").show()


In [ ]:
spark.sql("""
    WITH order_comparison AS (
        SELECT 
            customer_id,
            order_id,
            order_date,
            sales_amount AS curr_sales,
            LAG(sales_amount, 1) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS prev_sales
        FROM Orders
    )
    SELECT 
        customer_id,
        order_id,
        order_date,
        prev_sales,
        curr_sales,
        ROUND(prev_sales - curr_sales, 2) AS revenue_drop
    FROM order_comparison
    WHERE prev_sales IS NOT NULL AND curr_sales < prev_sales
""").show()


In [18]:
spark.sql("""
    SELECT 
        DATE_FORMAT(order_date, 'EEEE') AS day_of_week,
        ROUND(SUM(sales_amount), 2) AS total_revenue
    FROM Orders
    GROUP BY DATE_FORMAT(order_date, 'EEEE')
    ORDER BY total_revenue DESC
""").show()


+-----------+-------------+
|day_of_week|total_revenue|
+-----------+-------------+
|     Monday|    5655873.8|
|   Saturday|   5031840.67|
|  Wednesday|   4647350.79|
|    Tuesday|   4534374.49|
|     Sunday|   4483950.16|
|   Thursday|   4423944.61|
|     Friday|   4280333.46|
+-----------+-------------+

